<a href="https://colab.research.google.com/github/SoyebAhmed5/flyrank-internship-ml/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SoyebAhmed5/flyrank-internship-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*
### Ranked Content Actions and Reason Codes

The purpose of this section is to convert the validated model output into a ranked content review queue.

The Logistic Regression model produces a score representing its estimated likelihood that a content item belongs to the declining-content class.

Content items are ranked from the highest model score to the lowest. This ranking helps a content team decide which items to review first.

Each item also receives reason codes based on available metadata, such as content age and time since the last update.

The playbook uses the following action categories:

- HIGH_RISK_REVIEW: Items with higher model scores that should receive earlier human review.
- REFRESH_REVIEW: Older content that has not been updated recently and may need a freshness check.
- CONTENT_REVIEW: Items that deserve a general review of relevance, quality, and search intent.
- MONITOR: Items with lower model scores that can remain in the monitoring queue.

The reason codes are decision-support signals, not proof that a particular issue caused declining performance.

The ranking is intended to prioritize human attention. It does not automatically authorize content updates or guarantee that a recommended action will improve traffic.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

# Load the dataset
url = (
    "https://raw.githubusercontent.com/"
    "SoyebAhmed5/flyrank-internship-ml/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(url)

# Prepare the target
model_df = df.dropna(
    subset=["trend_direction", "client_id"]
).copy()

model_df["target_declining"] = (
    model_df["trend_direction"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("down")
    .astype(int)
)

print("Dataset loaded:", df.shape)
print("Model dataset:", model_df.shape)
print("Declining-content rate:", round(
    model_df["target_declining"].mean(), 4
))

Dataset loaded: (30000, 44)
Model dataset: (30000, 45)
Declining-content rate: 0.5421


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

# Features used in ML-09
numeric_features = [
    "content_age_days",
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count"
]

categorical_features = [
    "content_type",
    "main_intent",
    "competition_level"
]

feature_columns = numeric_features + categorical_features

X = model_df[feature_columns]
y = model_df["target_declining"]
groups = model_df["client_id"]

# Client-grouped split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

# Preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Logistic Regression
group_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

# Train and predict
group_model.fit(X_train_group, y_train_group)

group_prob = group_model.predict_proba(
    X_test_group
)[:, 1]

print("Grouped model trained successfully!")
print("Evaluation rows:", len(X_test_group))
print("Predictions generated:", len(group_prob))

Grouped model trained successfully!
Evaluation rows: 6163
Predictions generated: 6163


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Reason Codes and Archetype-to-Action Mapping

The playbook uses a combination of model scores and content metadata to organize the review queue.

| Archetype | Reason | Suggested action |
|---|---|---|
| STALE_CONTENT | Older content with a longer time since its last update | Review outdated information and consider a refresh |
| YOUNG_CONTENT | Recently created content | Check indexing, relevance, and early performance |
| GENERAL_REVIEW | Content that does not match the other categories | Review quality, search intent, and performance trends |

Model score thresholds of 0.50 and 0.65 are used as provisional operational prioritization rules. They have not been established as optimal business thresholds or validated decision boundaries.

The content age and update-time thresholds are also practical review heuristics, not proven causes of declining performance.

The model score determines the ranking, while metadata provides additional context for human reviewers.

The recommended actions are suggestions only. A content specialist must examine the actual page and its business context before deciding what to change.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

# -----------------------------------------
# 1. Prepare the ranked review queue
# -----------------------------------------

# Reuse the held-out client-grouped predictions from ML-09.
queue = model_df.loc[X_test_group.index].copy()

queue["decline_risk_score"] = group_prob

# Sort from highest score to lowest.
queue = queue.sort_values(
    "decline_risk_score",
    ascending=False
).copy()

queue["rank"] = range(1, len(queue) + 1)


# -----------------------------------------
# 2. Assign priority levels
# -----------------------------------------

def assign_priority(score):
    if score >= 0.65:
        return "HIGH"
    elif score >= 0.50:
        return "MEDIUM"
    else:
        return "LOW"


queue["priority"] = queue["decline_risk_score"].apply(
    assign_priority
)


# -----------------------------------------
# 3. Identify content archetypes
# -----------------------------------------

# These are operational categories, not learned
# or validated model classes.

queue["content_age_days"] = pd.to_numeric(
    queue["content_age_days"],
    errors="coerce"
)

queue["days_since_last_update"] = pd.to_numeric(
    queue["days_since_last_update"],
    errors="coerce"
)

def assign_archetype(row):

    age = row["content_age_days"]
    update_days = row["days_since_last_update"]

    if (
        pd.notna(age)
        and pd.notna(update_days)
        and age >= 180
        and update_days >= 90
    ):
        return "STALE_CONTENT"

    elif pd.notna(age) and age < 90:
        return "YOUNG_CONTENT"

    else:
        return "GENERAL_REVIEW"


queue["archetype"] = queue.apply(
    assign_archetype,
    axis=1
)


# -----------------------------------------
# 4. Generate reason codes
# -----------------------------------------

def generate_reason_codes(row):

    reasons = []

    if row["decline_risk_score"] >= 0.65:
        reasons.append("HIGH_MODEL_SCORE")

    elif row["decline_risk_score"] >= 0.50:
        reasons.append("MODERATE_MODEL_SCORE")

    if row["archetype"] == "STALE_CONTENT":
        reasons.append("OLD_AND_NOT_RECENTLY_UPDATED")

    if row["archetype"] == "YOUNG_CONTENT":
        reasons.append("RECENT_CONTENT")

    if not reasons:
        reasons.append("LOWER_MODEL_SCORE")

    return ", ".join(reasons)


queue["reason_codes"] = queue.apply(
    generate_reason_codes,
    axis=1
)


# -----------------------------------------
# 5. Map archetypes to suggested actions
# -----------------------------------------

action_mapping = {
    "STALE_CONTENT":
        "Review outdated information and consider a content refresh.",

    "YOUNG_CONTENT":
        "Check indexing, relevance, and early performance before making changes.",

    "GENERAL_REVIEW":
        "Review content quality, search intent, and performance trends."
}

queue["suggested_action"] = queue["archetype"].map(
    action_mapping
)


# -----------------------------------------
# 6. Create final review queue
# -----------------------------------------

# Keep only the fields needed for the playbook.
# No client identifiers or private URLs are exported.

queue_columns = [
    "content_id",
    "rank",
    "priority",
    "decline_risk_score",
    "archetype",
    "reason_codes",
    "suggested_action"
]

ranked_queue = queue[queue_columns].copy()

ranked_queue["decline_risk_score"] = (
    ranked_queue["decline_risk_score"].round(4)
)

print("Ranked content review queue generated!")
print("Total content items:", len(ranked_queue))

display(ranked_queue.head(15))

Ranked content review queue generated!
Total content items: 6163


,content_id,rank,priority,decline_risk_score,archetype,reason_codes,suggested_action
20353,content_035b17df40a9,1,HIGH,0.6892,GENERAL_REVIEW,HIGH_MODEL_SCORE,"Review content quality, search intent, and per..."
6897,content_7bf733e787cb,2,HIGH,0.6865,GENERAL_REVIEW,HIGH_MODEL_SCORE,"Review content quality, search intent, and per..."
7264,content_b35a7acdc3a8,3,HIGH,0.6830,GENERAL_REVIEW,HIGH_MODEL_SCORE,"Review content quality, search intent, and per..."
2555,content_29717601b959,4,HIGH,0.6766,GENERAL_REVIEW,HIGH_MODEL_SCORE,"Review content quality, search intent, and per..."
15277,content_c63e0af4b14a,5,HIGH,0.6756,GENERAL_REVIEW,HIGH_MODEL_SCORE,"Review content quality, search intent, and per..."
16607,content_27a2fd237adb,6,HIGH,0.6740,GENERAL_REVIEW,HIGH_MODEL_SCORE,"Review content quality, search intent, and per..."
229,content_5b377954b9b8,7,HIGH,0.6727,GENERAL_REVIEW,HIGH_MODEL_SCORE,"Review content quality, search intent, and per..."
14521,content_5888602cf5d2,8,HIGH,0.6714,GENERAL_REVIEW,HIGH_MODEL_SCORE,"Review content quality, search intent, and per..."
23704,content_84bc4706ba53,9,HIGH,0.6697,GENERAL_REVIEW,HIGH_MODEL_SCORE,"Review content quality, search intent, and per..."
13218,content_8025e3894512,10,HIGH,0.6685,GENERAL_REVIEW,HIGH_MODEL_SCORE,"Review content quality, search intent, and per..."


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check queue quality

print("Priority distribution:")
display(
    ranked_queue["priority"].value_counts().to_frame("Count")
)

print("\nArchetype distribution:")
display(
    ranked_queue["archetype"].value_counts().to_frame("Count")
)

print("\nReason code distribution:")
display(
    ranked_queue["reason_codes"].value_counts().to_frame("Count")
)

# Basic validation checks
assert ranked_queue["rank"].is_unique
assert ranked_queue["decline_risk_score"].between(0, 1).all()
assert ranked_queue["content_id"].is_unique

print("\nAll queue checks passed!")

Priority distribution:


,Count
priority,
MEDIUM,3111
LOW,2927
HIGH,125



Archetype distribution:


,Count
archetype,
GENERAL_REVIEW,5094
STALE_CONTENT,1069



Reason code distribution:


,Count
reason_codes,
MODERATE_MODEL_SCORE,2700
LOWER_MODEL_SCORE,2269
OLD_AND_NOT_RECENTLY_UPDATED,658
"MODERATE_MODEL_SCORE, OLD_AND_NOT_RECENTLY_UPDATED",411
HIGH_MODEL_SCORE,125



All queue checks passed!


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.